# LeetCode 83 — Remove Duplicates from Sorted List
Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Linked List · con trỏ · thay đổi liên kết  
**Bài gốc:** https://leetcode.com/problems/remove-duplicates-from-sorted-list/

## Bạn sẽ học được gì?
1. Tự tạo và duyệt Linked List với ListNode, head, next, None.
2. Hiểu vì sao danh sách đã sắp xếp giúp phát hiện trùng nhau.
3. So sánh cách tạo danh sách mới với xóa node tại chỗ.
4. Trace từng lần sửa liên kết next và test mọi edge case.
5. Tự nộp deleteDuplicates đúng kiểu dữ liệu.

**Cách học:** đọc Markdown → đoán kết quả của cell code tiếp theo → chạy cell → so sánh. Phần TODO chỉ để luyện tập, không tự chạy bài tập chưa hoàn thiện.

## 1. Đề bài — chỉ giữ một node cho mỗi giá trị

Cho `head` của **singly linked list** (danh sách liên kết đơn) đã **sắp xếp không giảm**. Hãy loại bỏ các node có giá trị lặp, giữ **một node của mỗi giá trị**, trả về `head` của danh sách kết quả.

| Input (viết dạng mũi tên) | Output |
|---|---|
| `1 → 1 → 2` | `1 → 2` |
| `1 → 1 → 2 → 3 → 3` | `1 → 2 → 3` |
| rỗng `None` | `None` |

**Phân biệt bài #82:** bài #83 chỉ xóa các **bản sao thừa**, vẫn **giữ 1 node** cho mỗi giá trị xuất hiện; không xóa sạch nhóm trùng.

## 2. `ListNode` không phải list Python

Một **node** có 2 ô:
- `val` là giá trị, ví dụ 7.
- `next` là liên kết (reference) đến **node tiếp theo** hoặc `None` nếu là node cuối.

Ví dụ `a → b → c → None` không phải ba phần tử liên tiếp trong list Python: mỗi node lưu riêng một con trỏ. `head` là **địa chỉ của node đầu tiên**, không nhất thiết là chính một danh sách giá trị.

LeetCode cung cấp `ListNode` khi nộp; ở notebook ta tự định nghĩa để chạy minh họa.

In [ ]:
from __future__ import annotations
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next: Optional["ListNode"] = None):
        self.val = val
        self.next = next

first = ListNode(1)
second = ListNode(2)
first.next = second
print("head.val:",first.val)
print("head.next.val:",first.next.val)
print("node cuối next có phải None?", second.next is None)

## 3. Python nền tảng: `.`, `=`, `is None` và `while`

- `node.val`: lấy giá trị của node.
- `node.next`: lấy node kế tiếp, **không phải** giá trị kế tiếp.
- `node.next = other`: thay đổi **mũi tên** liên kết, không xóa đi một ô trong list Python.
- `node is None`: kiểm tra node có tồn tại không. Dùng `while node is not None:` để đi từng node.

`current = head` **không tạo bản sao**; hai biến chỉ cùng một node.

In [ ]:
head = ListNode(2, ListNode(5, ListNode(8)))
current = head
while current is not None:
    print("Đang ở node giá trị:", current.val)
    current = current.next
print("Đến cuối:",current)

alias = head
alias.val = 99
print("Sửa alias cũng làm head đổi:",head.val)

## 4. Hai hàm phụ chỉ để minh họa và test

`build_list([1,1,2])` tạo linked list từ list Python; `to_values(head)` đi theo các `next` để trả danh sách giá trị **cho dễ quan sát**. Trong LeetCode, không được thay `head` bằng list Python khi nộp.

In [ ]:
def build_list(values: list[int]) -> Optional[ListNode]:
    head = None
    for value in reversed(values):
        head = ListNode(value, head)
    return head

def to_values(head: Optional[ListNode]) -> list[int]:
    out = []
    node = head
    while node is not None:
        out.append(node.val)
        node = node.next
    return out

sample = build_list([1,1,2,3,3])
print("Duyệt linked list:",to_values(sample))

## 5. Vì sao chỉ cần so hai node đứng cạnh?

Vì danh sách **đã sắp xếp**, hai node bằng giá trị luôn nằm trong **cùng một cụm liên tiếp**. Ví dụ `1,1,2,3,3`: chỉ cần so `current.val` và `current.next.val`.

Nếu bằng nhau → node kế tiếp bị trùng, có thể bỏ qua. Nếu khác → chuyển `current` tới node kế tiếp.

Đừng dùng quy tắc này cho linked list **chưa sắp xếp**: `1→2→1` có hai giá trị 1 không kề nhau.

In [ ]:
nums = [1,1,2,3,3]
for i in range(len(nums)-1):
    print(f"{nums[i]} và {nums[i+1]} -> trùng?",nums[i]==nums[i+1])

## 6. Baseline: tạo danh sách mới (dễ suy nghĩ)

Một cách là đi qua đầu vào, chỉ đưa các **giá trị mới** vào list Python, rồi xây Linked List khác. Bản này dễ kiểm chứng nhưng cần bộ nhớ **O(n)** để giữ giá trị và tạo node mới.

Ta nên học cách giữ node cũ để có thể giải **in-place (tại chỗ)**.

In [ ]:
def dedup_new_nodes(head: Optional[ListNode]) -> Optional[ListNode]:
    values = []
    current = head
    while current is not None:
        if not values or values[-1] != current.val:
            values.append(current.val)
        current = current.next
    return build_list(values)

for data in ([1,1,2],[1,1,2,3,3],[],[7,7,7]):
    print(data,"->",to_values(dedup_new_nodes(build_list(data))))

## 7. Insight: nối bỏ qua node trùng thay vì tạo node mới

Giả sử `current` đang trỏ đến `1` đầu tiên trong `1a→1b→2`:

- Trước: `current.next` là node `1b`.
- Sau khi viết `current.next = current.next.next`, mũi tên từ `1a` nhảy thẳng tới node `2`.
- `current` **vẫn là `1a`**, cần so tiếp nếu có nhiều node trùng liên tiếp.

Ngược lại, khi hai node khác nhau, **giữ** liên kết và bước tiếp: `current = current.next`.

In [ ]:
a_node = ListNode(1)
b_node = ListNode(1)
c_node = ListNode(2)
a_node.next = b_node
b_node.next = c_node
print("Trước:",to_values(a_node))
a_node.next = a_node.next.next
print("Sau  :",to_values(a_node))
print("a_node vẫn là node cũ?",a_node.val==1)

## 8. `and` kiểm tra ngắn mạch — bảo vệ khỏi None

`while current is not None and current.next is not None:` có nghĩa là **chỉ** đọc `current.next` khi `current` đã được xác nhận tồn tại. Python dừng xét `and` ngay tại điều kiện sai đầu tiên (*short-circuit*).

Nếu viết `current.next.val` mà `current.next` là `None`, ta sẽ gặp `AttributeError`.

In [ ]:
empty = None
print("None được bảo vệ bởi and:", empty is not None and empty.next is not None)
node = ListNode(3)
print("Node cuối còn next?", node.next is not None)
# KHÔNG chạy node.next.val khi node.next == None

## 9. Thuật toán chính — sửa liên kết tại chỗ

1. Khởi tạo `current=head`.
2. Trong khi có `current` và `current.next`, so hai giá trị.
3. Nếu trùng, **bỏ qua node sau** bằng cách gán `.next`.
4. Nếu khác, dịch `current` tới node sau.
5. Trả lại **head ban đầu** (có thể là `None`).

In [ ]:
def dedup_inplace(head: Optional[ListNode]) -> Optional[ListNode]:
    current = head
    while current is not None and current.next is not None:
        if current.val == current.next.val:
            current.next = current.next.next
        else:
            current = current.next
    return head

head = build_list([1,1,2,3,3])
original_first_node = head
new_head = dedup_inplace(head)
print("Kết quả:", to_values(new_head))
print("Giữ nguyên node đầu:", new_head is original_first_node)

## 10. Bảng trace bằng tay: `1 → 1 → 2 → 3 → 3`

| Lượt | current | current.next (trước) | Quyết định | Danh sách nhìn từ head sau |
|---:|---:|---:|---|---|
| 1 | 1 | 1 | trùng → bỏ node 1 sau | 1→2→3→3 |
| 2 | 1 | 2 | khác → bước đến node 2 | 1→2→3→3 |
| 3 | 2 | 3 | khác → bước đến node 3 | 1→2→3→3 |
| 4 | 3 | 3 | trùng → bỏ node 3 sau | 1→2→3 |

Cuối cùng `current.next` bằng `None`, vòng lặp dừng.

In [ ]:
def trace_dedup(head: Optional[ListNode]):
    current=head
    step=0
    print("ĐẦU VÀO:",to_values(head))
    while current is not None and current.next is not None:
        step+=1
        a,b=current.val,current.next.val
        if a==b:
            current.next=current.next.next
            action="bỏ qua node trùng"
        else:
            current=current.next
            action="chuyển current sang node sau"
        print(f"lượt {step}: so {a} và {b} -> {action} -> {to_values(head)}")
    print("TRẢ VỀ head:",to_values(head))
    return head

trace_dedup(build_list([1,1,2,3,3]))

## 11. Chứng minh tính đúng — bất biến

Trước mỗi lượt, **đoạn từ head đến current** không còn các node có giá trị trùng kề nhau; ta đã giữ **một node đại diện** cho mỗi giá trị đi qua.

- Nếu node kế tiếp trùng, xóa liên kết tới node đó không làm mất **giá trị duy nhất** nào (vì current đã giữ giá trị ấy).
- Nếu khác, do đã sắp xếp, giá trị tiếp theo chưa xuất hiện ở đoạn đã xử lý, nên dịch current là an toàn.

Khi tới cuối, không còn cặp kề nhau trùng, tức mỗi giá trị chỉ còn một node.

## 12. Class Solution nộp LeetCode

Khi nộp, LeetCode định nghĩa `ListNode` sẵn. Dùng đúng tên `deleteDuplicates` và trả `Optional[ListNode]`. **Không** trả `list[int]` của `to_values`.

In [ ]:
class Solution:
    def deleteDuplicates(self, head: Optional[ListNode]) -> Optional[ListNode]:
        current = head
        while current is not None and current.next is not None:
            if current.val == current.next.val:
                current.next = current.next.next
            else:
                current = current.next
        return head

## 13. Kiểm thử: rỗng, 1 node, cụm trùng dài, số âm

Ta tạo linked list **mới cho từng test**, bởi thuật toán làm **biến đổi đầu vào** (mutation). Nếu dùng lại cùng object cho hai cách giải, test thứ hai có thể không còn kiểm thử dữ liệu ban đầu.

In [ ]:
cases_83 = [
    ([],[]), ([1],[1]), ([1,1,2],[1,2]),
    ([1,1,2,3,3],[1,2,3]), ([7,7,7,7],[7]),
    ([1,2,3],[1,2,3]), ([-2,-2,-1,0,0,3],[-2,-1,0,3]),
    ([1,1,1,2,2,2,3,3,3],[1,2,3]),
]
for values,expected in cases_83:
    linked = build_list(values)
    result = Solution().deleteDuplicates(linked)
    assert to_values(result) == expected, (values, expected)
    assert result is linked  # cùng node đầu, trừ trường hợp None
    assert to_values(dedup_new_nodes(build_list(values)))==expected
print("PASS",len(cases_83),"ca linked list")

## 14. Độ phức tạp — n là số node

- **Baseline tạo list mới:** Time O(n), Extra Space O(n) (chưa kể bộ nhớ của các node đầu vào).
- **Sửa liên kết tại chỗ:** Time O(n), Extra Space **O(1)** — chỉ giữ con trỏ `current`, không tạo node mới. Các node bị bỏ có thể được Python thu hồi khi không còn tham chiếu.

Mỗi lượt xử lý/loại bỏ ít nhất một node trong phần chưa xử lý; số lượt tối đa tỷ lệ với n.

## 15. Sai lầm thường gặp

1. `current = current.next` ngay cả sau khi xóa trùng → với `1→1→1`, có thể bỏ sót node trùng còn lại.
2. Truy cập `.next.val` khi `.next is None` → lỗi runtime.
3. Dùng `head = head.next` tùy tiện → dễ làm mất node đầu cần trả.
4. Chuyển linked list thành list Python rồi **trả list** → sai kiểu dữ liệu LeetCode.
5. Lẫn bài #82: xóa **mọi node thuộc nhóm trùng** không phải mục tiêu của #83.

In [ ]:
head = build_list([1,1,1,2])
trace_dedup(head)
print("Dù có 3 số 1 liên tiếp, chỉ giữ 1:",to_values(head))

## 16. Mở rộng pattern

Linked List **đã sắp xếp** → các giá trị bằng nhau nằm cạnh nhau → dùng **một con trỏ** so với node sau. Nếu đầu vào **chưa sắp xếp**, muốn giữ các giá trị duy nhất ta thường phải lưu giá trị đã thấy (`set`) hoặc sắp xếp lại (nếu được phép).

Tư duy quan trọng: thay đổi `.next` là **thay đổi cấu trúc liên kết**, không phải sửa giá trị của node.

## 17. Năm bài tập tự luyện

Các hàm TODO được định nghĩa nhưng không gọi. Đáp án ở **cuối notebook**.

### Bài 1 — Đọc node kế tiếp · Tự làm

Cho node không rỗng và có next, trả giá trị của node kế tiếp.

**Gợi ý cấp 1 — tư duy:** Truy cập qua liên kết.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng `.next.val`.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

return node.next.val.

</details>

In [ ]:
def exercise_83_1(node: ListNode) -> int:
    # TODO: node có next hợp lệ
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_83_1(build_list([2,5])) == 5

### Bài 2 — Đếm số node · Tự làm

Viết hàm đếm các node từ head đến None.

**Gợi ý cấp 1 — tư duy:** Đi theo .next cho tới khi hết.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

while current is not None.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

count+=1 rồi current=current.next.

</details>

In [ ]:
def exercise_83_2(head: Optional[ListNode]) -> int:
    # TODO: đếm số node
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_83_2(build_list([1,2,3])) == 3

### Bài 3 — Tính tay · Tự làm

Từ `[1,1,1,2,2,4]`, trả list giá trị sau khử trùng.

**Gợi ý cấp 1 — tư duy:** Các số trùng nằm sát nhau.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Giữ phần tử đầu mỗi nhóm.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

[1,2,4]

</details>

In [ ]:
def exercise_83_3() -> list[int]:
    # TODO: tính bằng tay
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_83_3() == [1,2,4]

### Bài 4 — Tự viết xóa trùng in-place · Tự làm

Viết lại thuật toán xóa node trùng bằng `.next` mà không gọi `dedup_inplace`.

**Gợi ý cấp 1 — tư duy:** So current.val với current.next.val.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng while có hai điều kiện khác None.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Trùng: current.next=current.next.next; khác: current=current.next.

</details>

In [ ]:
def exercise_83_4(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO: in-place
    pass

# Tự mở comment sau khi hoàn thiện:
# assert to_values(exercise_83_4(build_list([1,1,2,3,3]))) == [1,2,3]

### Bài 5 — Biến thể: đếm số GIÁ TRỊ khác nhau · Tự làm

Không sửa linked list; trả số lượng giá trị phân biệt trong danh sách đã sắp xếp.

**Gợi ý cấp 1 — tư duy:** Đếm node đầu của mỗi nhóm trùng.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

So node trước với node hiện tại.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

count=0; prev=None; duyệt và tăng count khi khác prev.

</details>

In [ ]:
def exercise_83_5(head: Optional[ListNode]) -> int:
    # TODO: đếm giá trị phân biệt
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_83_5(build_list([1,1,2,3,3])) == 3

## 18. Đáp án tham khảo — chỉ đọc sau khi tự làm

In [ ]:
def answer_83_1(node): return node.next.val

def answer_83_2(head):
    count,cur=0,head
    while cur is not None:
        count+=1
        cur=cur.next
    return count

def answer_83_3(): return [1,2,4]

def answer_83_4(head):
    cur=head
    while cur is not None and cur.next is not None:
        if cur.val==cur.next.val:
            cur.next=cur.next.next
        else:
            cur=cur.next
    return head

def answer_83_5(head):
    count,cur=0,head
    previous=None
    while cur is not None:
        if previous is None or cur.val != previous:
            count+=1
        previous=cur.val
        cur=cur.next
    return count

assert answer_83_1(build_list([2,5]))==5
assert answer_83_2(build_list([1,2,3]))==3
assert answer_83_3()==[1,2,4]
assert to_values(answer_83_4(build_list([1,1,2,3,3])))==[1,2,3]
assert answer_83_5(build_list([1,1,2,3,3]))==3
print("PASS tất cả đáp án bài tập #83")

## Tóm tắt — mẫu tư duy cần nhớ

**Sorted Linked List** → giá trị trùng nằm cạnh nhau → so `current.val` và `current.next.val` → nếu trùng, **nối bỏ qua node sau**; nếu khác, bước tới node sau → **Time O(n), extra space O(1)**.

## Checklist trước khi submit

- [ ] Tôi phân biệt `ListNode` với list Python.
- [ ] Tôi giải thích được `current.next = current.next.next`.
- [ ] Tôi không di chuyển current ngay sau khi xóa trùng.
- [ ] Tôi đã test head=None, một node, toàn số giống nhau, trùng ở cuối.
- [ ] Tôi trả `head` (kiểu ListNode/None), không phải list[int].
- [ ] Tôi nộp `Solution.deleteDuplicates`.